# Running Inference on ONNX Models Quantized with the Model Compression Toolkit (MCT)

[Run this tutorial in Google Colab](https://colab.research.google.com/github/SonySemiconductorSolutions/mct-model-optimization/blob/main/tutorials/notebooks/mct_features_notebooks/pytorch/example_pytorch_onnx_inference.ipynb)

## Overview
This tutorial demonstrates how to run inference with ONNX Runtime on a PyTorch model that has been quantized with the Model Compression Toolkit (MCT) and exported to ONNX. It covers two scenarios: a simple model built only from standard PyTorch layers, and a simple model that also contains a CustomLayer (`MulticlassNMS` from `edgemdt_cl`). Note that the models used here are small and randomly initialized for demonstration purposes only, so the inference results themselves are not meaningful.

## Summary:
In this tutorial, we will cover:

1. Building a simple PyTorch model without a CustomLayer, quantizing it with MCT, exporting it to ONNX, and running inference with ONNX Runtime.
2. Building a simple PyTorch model that includes a CustomLayer, quantizing it with MCT, exporting it to ONNX, and running inference with ONNX Runtime.

## Setup
Install the relevant packages:

In [ ]:
!pip install -q torch==2.6.0 torchvision==0.21.0
!pip install -q onnx==1.17.0 onnxruntime==1.21.1 onnxruntime-extensions==0.13.0

Install the Model Compression Toolkit. Note that `edgemdt_cl`, which provides the CustomLayer used later in this tutorial, is installed automatically as a dependency of MCT.

In [ ]:
import importlib
if not importlib.util.find_spec('model_compression_toolkit'):
    !pip install model_compression_toolkit

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import model_compression_toolkit as mct
import onnxruntime as ort
import mct_quantizers as mctq

## Part 1: Inference Without a CustomLayer
In this part, we build a simple PyTorch model made only of standard layers, quantize it with MCT, export it to ONNX, and run inference with ONNX Runtime.

### Create a Simple Float Model
Let's create a small CNN classifier for demonstration purposes.

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Linear(16, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


float_model = SimpleModel()

### Quantize the Model with MCT
Notice that here the representative dataset is random for demonstration only.

In [ ]:
def representative_data_gen():
    yield [np.random.random((1, 3, 32, 32)).astype(np.float32)]


quantized_model, _ = mct.ptq.pytorch_post_training_quantization(float_model, representative_data_gen=representative_data_gen)

### Export the Quantized Model to ONNX

In [ ]:
onnx_file_path = 'model_without_custom_layer.onnx'

mct.exporter.pytorch_export_model(
    model=quantized_model,
    save_model_path=onnx_file_path,
    repr_dataset=representative_data_gen)

### Run Inference with ONNX Runtime
To load an ONNX model exported in MCTQ format and perform inference with it, use the `get_ort_session_options` method from `mct_quantizers` when creating an ONNX Runtime session.

In [ ]:
import onnxruntime as ort
import mct_quantizers as mctq

sess = ort.InferenceSession(onnx_file_path,
                            mctq.get_ort_session_options(),
                            providers=['CPUExecutionProvider'])

_input_data = next(representative_data_gen())[0]
_model_output_name = sess.get_outputs()[0].name
_model_input_name = sess.get_inputs()[0].name

# Run inference
predictions = sess.run([_model_output_name], {_model_input_name: _input_data})
print(predictions[0].shape)

## Part 2: Inference With a CustomLayer
In this part, we build a simple PyTorch model that includes a CustomLayer (`MulticlassNMS` from `edgemdt_cl`), quantize it with MCT, export it to ONNX, and run inference with ONNX Runtime.

In [ ]:
from edgemdt_cl.pytorch import MulticlassNMS

### Create a Simple Float Model with a CustomLayer
Let's create a small model made of a CNN backbone followed by two heads that produce box and class-score tensors, and a `MulticlassNMS` CustomLayer that post-processes them.

In [ ]:
class SimpleDetectionModel(nn.Module):
    def __init__(self, num_classes=2, max_detections=20, score_threshold=0.001, iou_threshold=0.7):
        super().__init__()
        self.num_classes = num_classes
        self.max_detections = max_detections

        self.backbone = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2))

        self.box_head = nn.Conv2d(8, 4 * max_detections, kernel_size=1)
        self.class_head = nn.Conv2d(8, num_classes * max_detections, kernel_size=1)
        self.nms = MulticlassNMS(score_threshold=score_threshold, iou_threshold=iou_threshold, max_detections=max_detections)

    def forward(self, x):
        batch = x.size(0)
        features = self.backbone(x)
        h, w = features.shape[2], features.shape[3]

        boxes = self.box_head(features).view(batch, self.max_detections, 4, h * w).mean(dim=3)
        scores = self.class_head(features).view(batch, self.max_detections, self.num_classes, h * w).mean(dim=3)
        scores = torch.softmax(scores, dim=2)

        nms_out = self.nms(boxes, scores)
        return nms_out.boxes, nms_out.scores, nms_out.labels


float_model_with_custom_layer = SimpleDetectionModel()

### Quantize the Model with MCT
As before, the representative dataset is random for demonstration only.

In [ ]:
def representative_data_gen_with_custom_layer():
    yield [np.random.random((1, 3, 32, 32)).astype(np.float32)]


quantized_model_with_custom_layer, _ = mct.ptq.pytorch_post_training_quantization(
    float_model_with_custom_layer,
    representative_data_gen=representative_data_gen_with_custom_layer)

### Export the Quantized Model to ONNX

In [ ]:
onnx_file_path_with_custom_layer = 'model_with_custom_layer.onnx'

mct.exporter.pytorch_export_model(
    model=quantized_model_with_custom_layer,
    save_model_path=onnx_file_path_with_custom_layer,
    repr_dataset=representative_data_gen_with_custom_layer)

### Run Inference with ONNX Runtime
To load an ONNX model exported from a model that contains a CustomLayer and run inference with it, import `mct_quantizers` and use the `load_custom_ops` method from `edgemdt_cl.pytorch` when creating an ONNX Runtime session.

In [ ]:
import onnxruntime as ort
import mct_quantizers as mctq
from edgemdt_cl.pytorch import load_custom_ops

sess = ort.InferenceSession(onnx_file_path_with_custom_layer,
                            load_custom_ops(),
                            providers=['CPUExecutionProvider'])

_input_data = next(representative_data_gen_with_custom_layer())[0]
_model_input_name = sess.get_inputs()[0].name

# Run inference
boxes, scores, labels = sess.run(output_names=None, input_feed={_model_input_name: _input_data})
print("boxes:", boxes.shape)
print("scores:", scores.shape)
print("labels:", labels.shape)

## Summary
In this tutorial, we quantized two simple PyTorch models with MCT, exported them to ONNX, and ran inference on them with ONNX Runtime: one built only from standard layers, and one that included a CustomLayer (`MulticlassNMS`). 

## Copyrights
Copyright 2026 Sony Semiconductor Solutions, Inc. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.